<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_96_graph_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🕸️ **Module 96 — Graph Neural Networks Deep Dive** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 14 · Theory & Responsibility (UDL deep-dives)


# Module 96 — Graph Neural Networks Deep Dive

> The original roadmap left M24 ("GNNs") as a TODO, and M87 (GraphRAG) used graphs as a *retrieval substrate* without explaining how to **learn on them**. This module closes both gaps, built around UDL Chapter 13's four notebooks (**Graph Representation, Graph Classification, Neighborhood Sampling, Graph Attention Networks**) and modernized to the 2026 GNN landscape: message-passing (**MPNN**), the four canonical layers (**GCN · GraphSAGE · GIN · GAT**), the **scaling tricks** that took GNNs from toy 3k-node benchmarks to industrial billion-edge graphs (**PinSAGE, NeighborSampling, Cluster-GCN, GraphSAINT**), and the **Graph Transformer** that beat purpose-built GNNs by 2024.

### What you'll cover
1. The graph data model — adjacency · features · tasks
2. Why classical NNs can't ingest graphs — permutation invariance
3. **GCN** (Kipf 2017) — the spectral → spatial bridge
4. **GraphSAGE** (Hamilton 2017) — neighborhood sampling for inductive learning
5. **GIN** (Xu 2019) — Weisfeiler-Lehman-equivalent expressive power
6. **GAT** (Veličković 2018) — attention over neighbors (UDL Chap13)
7. **MPNN framework** (Gilmer 2017) — one equation for every GNN
8. **Scaling tricks** — Cluster-GCN, GraphSAINT, PinSAGE
9. **Graph Transformers** — Graphormer · GraphGPS · TokenGT (2022-2024 SOTA)
10. The **2026 production stack** — PyTorch Geometric · DGL · OGB · GraphRAG bridge to M87


## 1 · The graph data model

A graph `G = (V, E, X)` has:
- `V` — N nodes
- `E` — edges (sparse adjacency matrix `A ∈ {0,1}^{N×N}` or weighted)
- `X ∈ ℝ^{N×D}` — per-node features (and optionally per-edge features `e_ij`)

Three task tiers:

| Task | Predict | Examples |
|---|---|---|
| **Node-level** | label per node | citation classification (Cora, CiteSeer), social-network role prediction, fraud per account |
| **Edge-level** | label per edge / pair | link prediction, recommender system (user-item), drug interaction |
| **Graph-level** | label per graph | molecule property prediction (HIV-active y/n), code-graph bug detection |

The **transductive** vs **inductive** split is crucial:
- **Transductive** — train + test on the *same* graph; embeddings learned per node. Cora / CiteSeer / PubMed.
- **Inductive** — generalize to *unseen* graphs or new nodes. Required for production (new users, new molecules). GraphSAGE was the first to support this cleanly.

> 📌 **Why graphs are everywhere.** Knowledge graphs (M87), social networks (Facebook, LinkedIn), recommender systems (Pinterest PinSAGE, Uber Eats), drug discovery (Atomic Graph + property), code (data-flow graph of AST), road networks (Google Maps ETAs), particle physics (LHC jet tagging), traffic forecasting, fraud detection, GraphRAG retrieval. The substrate is general because **relationships are general**.


## 2 · Why classical NNs can't ingest graphs

A graph has no canonical node ordering — relabeling nodes shouldn't change the prediction. An MLP that takes a fixed-dimensional vector input violates this **permutation invariance**.

```
Same graph, two orderings →  DIFFERENT MLP inputs  →  DIFFERENT predictions  ❌
```

A GNN layer must satisfy:
```
f(P·X, P·A·Pᵀ) = P · f(X, A)             for any permutation P
```

The trick: aggregate **only over the (unordered) neighborhood** of each node, then update that node's embedding. This is **message passing**.

```
For each node v:
    1. AGGREGATE  m_v = Σ_{u ∈ N(v)} ψ(h_u, h_v, e_{uv})         # permutation-symmetric
    2. UPDATE     h_v' = φ(h_v, m_v)                              # local
Stack L layers → receptive field grows L hops.
```

Every GNN — GCN, GraphSAGE, GIN, GAT, MPNN, Graph Transformer — is some choice of `ψ` (the message function), `Σ` (the aggregator), and `φ` (the update). UDL Chap13 builds the basic versions step by step.


## 3 · GCN — spectral roots, spatial implementation

**Graph Convolutional Network** (Kipf & Welling, ICLR 2017) is the simplest practical GNN.

```
H^(l+1) = σ( D̂^{-1/2} · Â · D̂^{-1/2} · H^(l) · W^(l) )
```

Where `Â = A + I` (add self-loops) and `D̂` is its diagonal degree matrix.

**Two ways to see it:**
1. **Spectral.** A first-order approximation of a localized graph Fourier filter (Defferrard 2016 ChebNet). The `D̂^{-1/2} Â D̂^{-1/2}` is the symmetric-normalized Laplacian.
2. **Spatial / message-passing.** Each node's new embedding = **mean of neighbors' (and self's) embeddings**, projected by `W`. This is the only intuition you need 99% of the time.

GCN with 2 layers on Cora (2708 nodes, 7 classes) hits ~81% test accuracy from ~140 labeled nodes. **Semi-supervised** because the unlabeled nodes still propagate features.


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, X, A_hat):           # A_hat = D^-1/2 (A+I) D^-1/2
        return self.W(A_hat @ X)            # one message-pass + projection

class GCN(nn.Module):
    def __init__(self, in_dim, hid, out_dim):
        super().__init__()
        self.g1 = GCNLayer(in_dim, hid)
        self.g2 = GCNLayer(hid, out_dim)
    def forward(self, X, A_hat):
        H = F.relu(self.g1(X, A_hat))
        H = F.dropout(H, p=0.5, training=self.training)
        return self.g2(H, A_hat)            # logits per node


## 4 · GraphSAGE — neighborhood sampling for scale

GCN passes messages over **the entire adjacency matrix** — `A_hat @ X` is `O(N²)` dense or `O(|E|)` sparse. Fine for Cora (2708 nodes) but impossible for Facebook (~3B users, ~300B edges).

**GraphSAGE** (Hamilton, Ying, Leskovec NIPS 2017) introduces two ideas that unlock industrial scale:

1. **Sample a fixed-size neighborhood** per node (e.g. 25 hop-1, 10 hop-2) rather than aggregating all neighbors. Constant per-node cost.
2. **Inductive** — learn an aggregator function rather than a per-node embedding, so new nodes can be embedded at test time without retraining.

```
h_v^(l+1) = σ( W · [h_v^(l); AGG({h_u^(l) : u ∈ Sample(N(v))})] )
```

Aggregator choices in the original paper: **mean**, **LSTM** (over a random permutation of neighbors), **max-pool**. Mean is the default; pool is best on benchmarks; LSTM is a relic.

> 🏭 **PinSAGE (Ying 2018, Pinterest).** GraphSAGE scaled to the **3-billion-node, 18-billion-edge** Pinterest graph. Random-walk sampling + importance-pooling + curriculum training. Production recommender on Pinterest from 2018 onward. The blueprint for every industrial GNN since.


## 5 · GIN — Weisfeiler-Lehman expressive power

**Graph Isomorphism Network** (Xu, Hu, Leskovec, Jegelka ICLR 2019) asks: *how expressive are GNNs?* Their answer formalizes a known result — **MPNNs are at most as expressive as the 1-WL graph-isomorphism test.** And GIN achieves that bound.

```
h_v^(l+1) = MLP^(l)( (1 + ε^(l)) · h_v^(l) + Σ_{u ∈ N(v)} h_u^(l) )
```

Two design choices:
1. **Sum aggregator** — strictly more expressive than mean or max (mean loses cardinality information; max loses multiplicity)
2. **MLP after aggregation** — gives an injective hash, achieving 1-WL power

GIN beats GCN/SAGE on most graph-classification benchmarks (TU Datasets, OGB-PPA). On node-classification it's roughly tied — different tasks favor different aggregators.

> 📐 **What "1-WL" means.** The 1-dimensional Weisfeiler-Lehman test is an iterative color-refinement algorithm that distinguishes most non-isomorphic graphs but **cannot** distinguish certain symmetric pairs (e.g. two non-isomorphic graphs that look identical under hop-1 neighborhood multiset hashing). To go beyond 1-WL you need **k-WL** GNNs (e.g. PPGN, k-GNN, Equivariant Subgraph Aggregation Networks, Graph Transformers with structural encodings).


## 6 · GAT — attention over neighbors (UDL Chap13)

**Graph Attention Network** (Veličković, Cucurull, Casanova, Romero, Liò, Bengio ICLR 2018). UDL Chap13's `13_4_Graph_Attention_Networks.ipynb` walks this one end-to-end.

Same message-passing skeleton, but neighbors get **learned weights** instead of uniform `1/deg`:

```
α_{vu} = softmax_u( a^T · [W·h_v ; W·h_u] )        # attention score
h_v'   = σ( Σ_{u ∈ N(v)} α_{vu} · W · h_u )         # weighted sum
```

Multi-head attention works exactly like transformers (M19/M20): run K independent heads, concatenate (intermediate layers) or average (final layer). **GAT is essentially a sparse Transformer where the only allowed attention edges are graph edges** — a key intellectual link to Chapter 8 below.

GAT v1 normalized attention with softmax over neighbors. **GATv2** (Brody 2022) showed v1's attention was actually static (rank-1), and fixed it by moving the activation: `α = softmax(a^T · LeakyReLU(W·[h_v; h_u]))`. GATv2 is the modern default.


In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=8):
        super().__init__()
        self.heads = heads
        self.W = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.a = nn.Parameter(torch.empty(heads, 2 * out_dim))
        nn.init.xavier_uniform_(self.a)
        self.out_dim = out_dim
    def forward(self, X, edge_index):                      # edge_index: 2 × E
        N = X.size(0)
        H = self.W(X).view(N, self.heads, self.out_dim)    # node-wise projection
        src, dst = edge_index
        eij = (torch.cat([H[src], H[dst]], dim=-1) * self.a).sum(-1)
        eij = F.leaky_relu(eij, 0.2)
        # segment-softmax over dst (each node's incoming edges)
        eij = eij - scatter_max(eij, dst)[0][dst]
        α = eij.exp() / scatter_add(eij.exp(), dst)[dst]
        out = scatter_add(α.unsqueeze(-1) * H[src], dst, dim=0)
        return out.flatten(1)


## 7 · MPNN — one equation that subsumes all of the above

**Message-Passing Neural Network** (Gilmer, Schoenholz, Riley, Vinyals, Dahl ICML 2017): the unified abstraction.

```
For each layer l:
  Message:  m_{vu}^(l) = M^(l)(h_v^(l), h_u^(l), e_{vu})
  Aggregate: m_v^(l) = Σ_{u ∈ N(v)} m_{vu}^(l)
  Update:   h_v^(l+1) = U^(l)(h_v^(l), m_v^(l))
After L layers, READOUT:
  y = R({h_v^(L) : v ∈ V})           # for graph-level tasks
```

Every GNN you'll meet is a specific choice:
- **GCN**: `M = h_u`, `U = σ(W · mean(...))`
- **GraphSAGE**: `M = h_u`, `U = σ(W·[h_v; mean(sample(M))])`
- **GIN**: `M = h_u`, `U = MLP((1+ε)h_v + sum(M))`
- **GAT**: `M = α·W·h_u`, `U = σ(sum(M))`
- **MPNN (full)**: `M, U` are general MLPs that include edge features

**Why this matters.** Once you have the framework, every paper since 2017 is "a new `M` or a new aggregator." It's the same intellectual hygiene as MLP → CNN → Transformer.


## 8 · Scaling — Cluster-GCN, GraphSAINT, NeighborSampling

GraphSAGE's neighborhood sampling already scales to billions of nodes. Two refinements help further:

| Method | Idea | When |
|---|---|---|
| **NeighborSampler** (PyG) | Sample k_l neighbors per layer; fan-out = `k_1 · k_2 · ... · k_L` | The default 2026 trainer |
| **Cluster-GCN** (Chiang 2019) | Partition graph (METIS) into clusters; mini-batch = a small set of clusters | Memory-efficient |
| **GraphSAINT** (Zeng 2020) | Subgraph sampling (node, edge, random-walk); reweight for unbiasedness | High accuracy on large graphs |
| **PinSAGE** (Ying 2018) | Random-walk-based importance sampling + curriculum | Production recommender |
| **ShaDow-GNN** (Zeng 2021) | Sample personalized PageRank subgraph per node; train independently | Highly parallel |
| **GAS / GNNAutoScale** (Fey 2021) | Cache stale historical embeddings to avoid recomputing all hops | Easy speedup |

**Memory math.** A 2-layer GCN over a 1B-node graph with average degree 50 needs `O(N · D²)` for weights (fine) but `O(N · D)` for activations (≈ 200 GB for D=200). The sampling-based methods trade some accuracy for `O(B · D)` activations per mini-batch.

> 📊 **OGB benchmark** (Open Graph Benchmark, Hu 2020) defines the standard large-scale GNN evals: **ogbn-arxiv** (170k nodes), **ogbn-products** (2.4M), **ogbn-papers100M** (111M nodes). PyG and DGL ship reference implementations of every method above against OGB.


## 9 · Graph Transformers — beating purpose-built GNNs

Around 2021-2022 a quiet shift happened: **Transformers with the right positional encoding beat purpose-built GNNs** on most molecular benchmarks.

| Method | Key idea |
|---|---|
| **Graphormer** (Ying 2021) | Standard Transformer + 3 graph-specific encodings: **centrality** (degree), **spatial** (shortest-path distance), **edge** (bond type along path) |
| **SAN** (Kreuzer 2021) | Spectral attention; Laplacian eigenvectors as positional encoding |
| **GraphGPS** (Rampášek 2022) | Hybrid: MPNN layer + Transformer layer per block; modular framework |
| **TokenGT** (Kim 2022) | Tokenize each node + edge; pure Transformer with structural input |
| **GRIT** (Ma 2023) | Relative attention with shortest-path bias; no positional encoding |
| **GraphCheck** (2024) | Diffusion + Transformer hybrid for OOD generalization |

**Why they win.** (a) Long-range dependencies — MPNNs lose information past 4-6 hops due to over-smoothing; Transformers attend globally. (b) Better optimization landscape. (c) Composability with text and image Transformers.

**The MPNN counter-argument.** Pure Graph Transformers lose: (a) explicit graph structure unless heavily encoded; (b) inductive bias when training data is small; (c) speed (`O(N²)` attention vs `O(|E|)` message passing). The 2026 picture: **hybrid GNN + Transformer** (GraphGPS, GraphMixer) is the production choice; pure-Transformer wins large-scale benchmarks; pure-GNN wins few-shot and dense-cluster regimes.

> 🧪 **OGB-LSC, the giant benchmark.** Graphormer-V2 won OGB-LSC PCQM4Mv2 (3.7M molecules) in 2022. Subsequent Graph Transformers + diffusion + RL hybrids dominate 2023-2026.


## 10 · The 2026 production stack — and back to GraphRAG

| Tool | Maintainer | Use |
|---|---|---|
| **PyTorch Geometric (PyG)** | Stanford / Matthias Fey | The de-facto Python GNN library; full PyTorch integration |
| **DGL** | Amazon / Tsinghua | Production-friendly; bigger native support for distributed training |
| **GraphScope / NebulaGraph / Neo4j** | Alibaba / Vesoft / Neo4j | Storage + query layer (also used in M87) |
| **OGB / OGB-LSC** | Stanford | Standard benchmarks |
| **TorchDrug / PyG-Temporal / DGL-LifeSci** | Tang / PyG / DGL | Domain-specific extensions (drug, temporal, life-sciences) |
| **PinSAGE / Twin-Tower / GraphSAGE** | Pinterest / Uber / TikTok | Production recommender architectures |
| **NeMo-Graph (NVIDIA)** | NVIDIA | GPU-accelerated end-to-end GNN training stack |
| **Graphormer / GraphGPS** | Microsoft / Mila | Transformer-flavored GNN reference impls |

**Decision tree:**

```
                          Node / edge / graph task?
                            │
        ┌───────────────────┼───────────────────┐
       node                edge                graph
        │                    │                    │
    big graph?            recsys?            small + few classes?
    ├─ yes → SAGE        ├─ yes → PinSAGE   ├─ yes → GIN
    └─ no  → GCN/GAT      └─ no  → GAT       └─ no  → Graphormer/GPS
```

**Bridge back to M87 GraphRAG.** GraphRAG builds an LLM-extracted knowledge graph and retrieves over it; M96 *learns* on graphs. The two are complementary — **GraphRAG-aware GNN** = learn embeddings on the KG, use them as reranker features at retrieval time. Pinterest, LinkedIn, and TikTok all run this hybrid: GNN-trained embedding → ANN retrieval → LLM reranker.

> 🔄 **The intellectual triangle: M19/M20 (Transformer attention) ↔ M83 (CNN convolution) ↔ M96 (GNN message passing).** All three are *equivariance / invariance-aware* feature aggregators differing only in what symmetry they respect: 1-D order (RNN), 2-D translation (CNN), graph permutation (GNN), sequence permutation + content (Transformer). Knowing them as one family is the maturity threshold of deep-learning intuition.

## ✅ Recap

- **GNNs respect graph permutation** by aggregating only over (unordered) neighborhoods → message passing.
- **GCN** = mean of neighbors + linear projection. The first practical GNN.
- **GraphSAGE** = neighborhood sampling for inductive, billion-edge scale. PinSAGE is the canonical production app.
- **GIN** = sum aggregator + MLP achieves 1-WL expressive power.
- **GAT / GATv2** = attention over neighbors; sparse Transformer over graph edges.
- **MPNN** = the unified abstraction; every GNN is a choice of `M`, `Σ`, `U`.
- **Scaling**: NeighborSampler (PyG default), Cluster-GCN, GraphSAINT, ShaDow-GNN, GAS.
- **Graph Transformers** (Graphormer, GraphGPS, TokenGT, GRIT) match/beat MPNNs on most benchmarks since 2022 — but hybrid is the production pick.
- **2026 stack**: PyTorch Geometric + DGL + OGB + NeMo-Graph. Bridge to M87 via GNN-aware GraphRAG.

Next: **M97 — Normalizing Flows + Likelihood Models**.
